In [1]:
import anndata
import scanpy as sc
import pandas as pd
import cell2sentence as cs
from cell2sentence.tasks import embed_cells
import os
import sys
from pathlib import Path
import logging
import tracemalloc
import linecache
import torch
torch.cuda.is_available()

/gpfs/commons/groups/compbio/projects/rf_condas/cell2sentence/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


False

In [10]:
ref = anndata.read_h5ad("/gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/LIBD/sce_DLPFC_annotated/ref_symbol.h5ad")
gene_name1 = ref.var_names
# id1 = ref.var["gene_id"]
data1 = pd.read_csv(file1, header = 0)
#data1["Unnamed: 0"] = data1["Unnamed: 0"].str.replace("\\..+", "", regex= True)
#data1 = data1.set_index("broad.cell.type")
#data1 = data1.set_index("broad_type")
#data1 = data1.set_index("cellType_broad_hc")
data1 = data1.set_index("Unnamed: 0")
#data1 = data1.T
#data1.columns = data1.columns.str.replace("\\..+", "", regex= True)
shared = [x for x in data1.columns if x in gene_name1.to_list()]
len(shared)

14853

In [11]:
#dict1 = dict(zip(id1, gene_name1))
#sharedgenes = [dict1[x] for x in shared]
sharedgenes = shared 
sharedgenes[0:2]

['LINC01128', 'LINC00115']

In [12]:
data1

,WASH7P,MIR6859-1,ENSG00000268903,ENSG00000269981,WASH9P,ENSG00000228463,MTND1P23,MTND2P28,MTCO1P12,ENSG00000278791,...,MT-ND4,MT-TH,MT-TS2,MT-TL2,MT-ND5,MT-ND6,MT-TE,MT-CYB,MT-TT,MT-TP
Unnamed: 0,,,,,,,,,,,,,,,,,,,,,
2107UNHS-0291_Br2720_Mid_Bulk,42,6,2,1,131,101,42,11974,2232,0,...,486390,0,0,2,87580,4416,94,304349,87,289
2107UNHS-0291_Br2720_Mid_Cyto,72,3,7,1,210,46,120,10983,2929,3,...,668026,0,0,2,116969,1234,16,449146,76,85
2107UNHS-0291_Br2720_Mid_Nuc,128,23,2,3,365,56,129,15999,6238,0,...,759033,2,0,0,135618,6973,100,390075,111,373
2107UNHS-0291_Br6432_Ant_Bulk,146,24,50,15,354,277,213,12721,6617,0,...,663541,1,0,0,112211,8791,122,408091,247,472
2107UNHS-0291_Br6432_Ant_Cyto,70,5,7,1,193,96,115,5428,2467,0,...,444694,0,0,0,67203,1055,9,311535,92,78
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
AN00000906_Br8667_Ant_Cyto,38,3,2,0,118,47,4,249,339,0,...,15110,12,39,5,9214,4404,252,17085,514,753
AN00000906_Br8667_Ant_Nuc,64,18,4,0,152,132,4,477,571,1,...,24409,6,85,7,17794,20063,2118,20453,148,1495
AN00000906_Br8667_Mid_Bulk,9,1,6,0,68,120,0,236,362,0,...,7013,0,0,0,3847,333,4,6637,163,216


In [16]:
file1= "/gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/LIBD/sce_DLPFC_annotated_signature_matrix_broad_symbol.csv"
c2s_save_dir1="/gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/LIBD/c2s/2bre2"
c2s_save_name1 = "sce_2bre"
#model_path1 = "/gpfs/commons/groups/compbio/projects/rf_projects/rf_packages/C2S-Pythia-410m-cell-type-prediction"
#model_path1 = "/gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/LIBD/c2s/410mre/2026-02-14-00_23_04_finetune_cell_type_prediction/checkpoint-27800"
model_path1 = "/gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/LIBD/c2s/2bre/2026-02-14-01_29_10_finetune_cell_type_prediction/checkpoint-56515/"
model_save_dir1 = "/gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/LIBD/c2s/2bre2"
model_save_name1 = "cell_embedding_prediction"

In [14]:
file1 = "/gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/LIBD/rse_symbol.csv"
c2s_save_dir1="/gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/LIBD/c2s/2bre"
c2s_save_name1 = "rse_2bre2"
#model_path1 = "/gpfs/commons/groups/compbio/projects/rf_projects/rf_packages/C2S-Pythia-410m-cell-type-prediction"
#model_path1 = "/gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/LIBD/c2s/410mre/2026-02-14-00_23_04_finetune_cell_type_prediction/checkpoint-27800"
model_path1 = "/gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/LIBD/c2s/2bre/2026-02-14-01_29_10_finetune_cell_type_prediction/checkpoint-56515/"
model_save_dir1 = "/gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/LIBD/c2s/2bre2"
model_save_name1 = "cell_embedding_prediction"

In [6]:
def display_top(snapshot, key_type='lineno', limit=10):
    snapshot = snapshot.filter_traces((
        tracemalloc.Filter(False, "<frozen importlib._bootstrap>"),
        tracemalloc.Filter(False, "<unknown>"),
    ))
    top_stats = snapshot.statistics(key_type)

    print("Top %s lines" % limit)
    for index, stat in enumerate(top_stats[:limit], 1):
        frame = stat.traceback[0]
        print("#%s: %s:%s: %.1f KiB"
              % (index, frame.filename, frame.lineno, stat.size / 1024))
        line = linecache.getline(frame.filename, frame.lineno).strip()
        if line:
            print('    %s' % line)

    other = top_stats[limit:]
    if other:
        size = sum(stat.size for stat in other)
        print("%s other: %.1f KiB" % (len(other), size / 1024))
    total = sum(stat.size for stat in top_stats)
    print("Total allocated size: %.1f KiB" % (total / 1024))
    return(total / 1024 / 1024)


In [7]:
def get_embedding_c2s(file,
                      c2s_save_dir, c2s_save_name, 
                      model_path, model_save_dir, model_save_name,
                      embedding_save_dir = None,
                      transpose = False,
                      index_name = "Unnamed: 0", 
                      gene_name_rm = "\\..+",
                      use_genes = None,
                      gene_name = None, reorder_obs_name = None, 
                      ngenes = 200,
                      log = True, log_path = None):
    # Set up log
    if log:
        logging.basicConfig(format='%(asctime)s %(message)s', level=logging.WARNING,
                            filename=os.path.dirname(file1) + "/c2s_embeddings_" + Path(file1).stem + ".log",
                            filemode='w', force = True)
        logger = logging.getLogger(__name__)
        sys.stdout.write = logger.info
        sys.stderr.write = logger.error
        logger.warning('Prep data')
        tracemalloc.start()
    # Load data
    data = pd.read_csv(file, header = 0)
    data = data.set_index(index_name)
    if transpose:
        data = data.T
    if gene_name_rm is not None:
        data.columns = data.columns.str.replace(gene_name_rm, "", regex= True)
        #data[index_name] = data[index_name].str.replace(gene_name_rm, "", regex= True)
    if use_genes is not None:
        data = data[use_genes]
        #data = data.loc[use_genes]
    data = data.loc[:, ~data.columns.duplicated()]
    adata = sc.AnnData(data)
    if gene_name is not None:
        adata.var_names = gene_name
    adata.obs["organism"]="Homo sapiens"
    adata.obs["cell_type"]="unknown"
    adata.obs["tissue"]="unknown"
    adata.obs["sex"]="unknown"
    adata.obs["batch_condition"]="unknown"
    adata_obs_cols_to_keep = ["organism","cell_type","tissue", "sex", "batch_condition"]

    # Create CSData object
    if log:
        logger.warning('Prep model')
    arrow_ds, vocabulary = cs.CSData.adata_to_arrow(
        adata=adata, 
        random_state=42, 
        sentence_delimiter=' ',
        label_col_names=adata_obs_cols_to_keep
    )
    csdata = cs.CSData.csdata_from_arrow(
        arrow_dataset=arrow_ds, 
        vocabulary=vocabulary,
        save_dir=c2s_save_dir,
        save_name=c2s_save_name,
        dataset_backend="arrow"
    )

    # Define CSModel object
    csmodel = cs.CSModel(
        model_name_or_path=model_path,
        save_dir=model_save_dir,
        save_name=model_save_name
    )

    # Get embeddings
    if log:
        logging.warning('Extract embedding')
    embedded_cells = embed_cells(
        csdata=csdata,
        csmodel=csmodel,
        n_genes=ngenes
    )
    df = pd.DataFrame(embedded_cells)
    if reorder_obs_name is None:
        df["name"] = adata.obs_names.to_list()
    else:
        df["name"] = ["sample_" + str(x) for x in list(range(1, df.shape[0]+1))]
    if log:
        logging.warning('Save embedding')
    if embedding_save_dir is None: 
        df.set_index('name').to_csv(os.path.dirname(file1) + "/c2s_embeddings_" + Path(file1).stem + ".csv")
    else:
        df.set_index('name').to_csv(embedding_save_dir + "/c2s_embeddings_" + c2s_save_name + ".csv")
    if log:
        logger.warning('Done')
        snapshot = tracemalloc.take_snapshot()
        mem=display_top(snapshot)
        logger.warning('Memory used: ' + str(mem) + " MiB")
        if torch.cuda.is_available():
            free, total = torch.cuda.mem_get_info()
            logger.warning('GPU Memory used: ' + str((total - free) / 1024 / 1024) + " MiB")

In [17]:
get_embedding_c2s(file1,
                  c2s_save_dir1, c2s_save_name1, 
                  model_path1, model_save_dir1, model_save_name1,
                  embedding_save_dir = model_save_dir1, ngenes = 200, #ranspose = True,
                  index_name = "cellType_broad_hc")#"Unnamed: 0", use_genes = shared, gene_name = sharedgenes)

In [10]:
adata = ref
adata.obs_keys
adata.obs["organism"]="Homo sapiens"
adata.obs["cell_type"]=adata.obs["broad_type"]
adata.obs["tissue"]="pbmc"
adata.obs["sex"]="?"
adata.obs["batch_condition"]="?"
adata_obs_cols_to_keep = ["organism","cell_type","tissue", "sex", "batch_condition"]
adata

AnnData object with n_obs × n_vars = 4800 × 22982
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'cell_ID', 'cell_ID.old', 'cell_type', 'nReads_SimBu', 'nGenes_SimBu', 'batch', 'broad_type', 'ident', 'organism', 'tissue', 'sex', 'batch_condition'
    var: 'gene_name', 'gene_id', 'gene_id_long'

In [6]:
arrow_ds, vocabulary = cs.CSData.adata_to_arrow(
    adata=adata, 
    random_state=42, 
    sentence_delimiter=' ',
    label_col_names=adata_obs_cols_to_keep
)

WARN: more variables (24049) than observations (14744)... did you mean to transpose the object (e.g. adata.T)?
WARN: more variables (24049) than observations (14744), did you mean to transpose the object (e.g. adata.T)?
100%|███████████████████████████████████| 14744/14744 [00:04<00:00, 3631.04it/s]


In [7]:
c2s_save_dir = "/gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/deconvBench/pbmc/c2s/finetune"  # C2S dataset will be saved into this directory
c2s_save_name = "hoek"  # This will be the name of our C2S dataset on disk
csdata = cs.CSData.csdata_from_arrow(
    arrow_dataset=arrow_ds, 
    vocabulary=vocabulary,
    save_dir=c2s_save_dir,
    save_name=c2s_save_name,
    dataset_backend="arrow"
)

Saving the dataset (1/1 shards): 100%|█| 14744/14744 [00:00<00:00, 154919.24 exa


In [8]:
# cell_type_prediction_model_path = "/gpfs/commons/groups/compbio/projects/rf_projects/rf_packages/C2S-Scale-Pythia-1b-pt"
cell_type_prediction_model_path = "/gpfs/commons/groups/compbio/projects/rf_projects/rf_packages/C2S-Pythia-410m-cell-type-prediction"
save_dir = "/gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/deconvBench/pbmc/c2s/finetune"
save_name = "cell_embedding_prediction_C2S-Scale-Pythia-410m_refinetune_hao"
csmodel = cs.CSModel(
    model_name_or_path=cell_type_prediction_model_path,
    save_dir=save_dir,
    save_name=save_name
)

Using device: cuda


In [9]:
from datetime import datetime
training_task = "cell_type_prediction"
datetimestamp = datetime.now().strftime('%Y-%m-%d-%H_%M_%S')
output_dir = os.path.join(csmodel.save_dir, datetimestamp + f"_finetune_{training_task}")
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
print(output_dir)

/gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/deconvBench/pbmc/c2s/finetune/2026-01-29-11_07_50_finetune_cell_type_prediction


In [10]:
del arrow_ds
del adata
import torch
torch.cuda.empty_cache()
import gc
gc.collect()

76

In [11]:
from transformers import TrainingArguments

train_args = TrainingArguments(
    bf16=True,
    fp16=False,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    learning_rate=1e-5,
    load_best_model_at_end=True,
    logging_steps=50,
    logging_strategy="steps",
    lr_scheduler_type="cosine",
    num_train_epochs=5, 
    eval_steps=50,
    evaluation_strategy="steps",
    save_steps=100,
    save_strategy="steps",
    save_total_limit=1,
    warmup_ratio=0.05,
    output_dir=output_dir,
    torch_empty_cache_steps=1
)
csmodel.fine_tune(
    csdata=csdata,
    task=training_task,
    train_args=train_args,
    loss_on_response_only=False,
    top_k_genes=200,
    max_eval_samples=500,
)

/gpfs/commons/groups/compbio/projects/rf_condas/cell2sentence/lib/python3.8/site-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead


Reloading model from path on disk: /gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/deconvBench/pbmc/c2s/finetune/cell_embedding_prediction_C2S-Scale-Pythia-410m_refinetune_hao


Map (num_proc=3): 100%|██████████| 14744/14744 [00:10<00:00, 1392.09 examples/s]
/gpfs/commons/groups/compbio/projects/rf_condas/cell2sentence/lib/python3.8/site-packages/cell2sentence/csmodel.py:210: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.


Starting training. Output directory: /gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/deconvBench/pbmc/c2s/finetune/2026-01-29-11_07_50_finetune_cell_type_prediction
Selecting 500 samples of eval dataset to shorten validation loop.


/gpfs/commons/groups/compbio/projects/rf_condas/cell2sentence/lib/python3.8/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.


Step,Training Loss,Validation Loss
50,1.377900,1.343639
100,1.319000,1.309132
150,1.288500,1.301438
200,1.294800,1.296570
250,1.288400,1.295528
300,1.290900,1.293559
350,1.302400,1.292082
400,1.294600,1.295599
450,1.289000,1.294467
500,1.284700,1.298671


/gpfs/commons/groups/compbio/projects/rf_condas/cell2sentence/lib/python3.8/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
/gpfs/commons/groups/compbio/projects/rf_condas/cell2sentence/lib/python3.8/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
/gpfs/commons/groups/compbio/projects/rf_condas/cell2sentence/lib/python3.8/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
/gpfs/commons/groups/compbio/projects/rf_condas/cell2sentence/lib/python3.8/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
/gpfs/commons/groups/compbio/pro

Finetuning completed. Updated model saved to disk at: /gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/deconvBench/pbmc/c2s/finetune/2026-01-29-11_07_50_finetune_cell_type_prediction


In [3]:
adata = ref
adata.obs_keys
adata.obs["organism"]="Homo sapiens"
adata.obs["cell_type"]=adata.obs["broad.cell.type"]
adata.obs["tissue"]="cortex"
adata.obs["sex"]="?"
adata.obs["batch_condition"]="?"
adata_obs_cols_to_keep = ["organism","cell_type","tissue", "sex", "batch_condition"]

In [ ]:
adata1 = sc.AnnData(data1)
adata1

In [ ]:
data1 = pd.read_csv(file1, header = 0)
data1 = data1.set_index("Unnamed: 0")
data1 = data1.T

In [ ]:
data1

In [ ]:
data1.columns = data1.columns.str.replace("\\..+", "", regex= True)
data1 = data1[shared]
data1

In [ ]:
data1 = data1.loc[:, ~data1.columns.duplicated()]

In [ ]:
shared = [x for x in data1.index.to_list() if x in id1.to_list()]